In [51]:
!pip install qiskit-ionq


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [52]:
# Updated imports
import random
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_ionq import IonQProvider
from typing import List, Tuple
from qiskit_aer import AerSimulator
import numpy as np
import os

# Initialize IonQ provider
# Set IONQ_API_KEY environment variable or pass directly
provider = IonQProvider("WAUL3EgxQmFvYSB52uDAMgHg1Du9HmtH")


# UNCOMMENT BELOW IF YOU WANT TO CHECK WHICH BACKENDS ARE AVAILABLE RIGHT NOW
# def test_backend_connection(backend_name): # Checks which backends are available right now
#     """Test if a specific backend works"""
#     try:
#         backend = provider.get_backend(backend_name)
#         print(f"✅ Backend '{backend_name}' exists")
        
#         # Test with minimal circuit
#         from qiskit import QuantumCircuit
#         qc = QuantumCircuit(1, 1)
#         qc.measure(0, 0)
        
#         print(f"   Testing with minimal circuit...")
#         job = backend.run(qc, shots=1)
#         result = job.result()
#         print(f"   ✅ '{backend_name}' works successfully!")
#         return True
        
#     except Exception as e:
#         print(f"   ❌ '{backend_name}' failed: {e}")
#         return False
# # Test all available backends
# print("Testing all available backends:")
# available_backends = provider.backends()
# for backend in available_backends:
#     test_backend_connection(backend.name)

# UNCOMMENT BELOW BASED ON WHCIH BACKEND YOU ARE WORKING WITH

backend = provider.get_backend("ionq_simulator")  # or "ionq_qpu" for real hardware
flag = 0 # flag to tell when it is in simulator vs. actual quantum hardware, 0 means simulator

# backend = provider.get_backend("ionq_qpu")  # or "ionq_qpu" for real hardware
# flag = 1 # flag to tell when it is in simulator vs. actual quantum hardware, 1 means quantum hardware

# backend = AerSimulator()
# # flag = 0 # flag to tell when it is in simulator vs. actual quantum hardware, 0 means AerSimulator


In [53]:
def random_bits(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def random_bases(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def prepare_batch_circuit(bits: List[int], bases: List[int]) -> QuantumCircuit:
    """Create a single circuit with multiple qubits for batch processing"""
    n = len(bits)
    qc = QuantumCircuit(n, n) # creates a quantum circuit with n qubits and n classical bits
    
    for i, (bit, basis) in enumerate(zip(bits, bases)):
        if bit == 1: # convert qubit to |1> by applying an X gate, this is necessary because by default qubits start in the |0> state
            qc.x(i)
        if basis == 1: # if basis is X, apply H to convert from |0> to |+> or from |1> to |->
            qc.h(i)
    
    return qc

def measure_batch_circuit(base_qc: QuantumCircuit, bases: List[int]) -> QuantumCircuit:
    """Add measurement operations for Bob's bases"""
    n = base_qc.num_qubits
    qc = base_qc.copy()
    
    for i, basis in enumerate(bases):
        if basis == 1:
            qc.h(i) # applies hadamard gate to qubit at index i
        qc.measure(i, i) # measures qubit in index i and stores it as a classical bit in index i
    
    return qc

def run_circuit_get_bits(qc: QuantumCircuit, shots: int = 1) -> List[int]:
    """Run circuit on IonQ and return measured bits"""
    # Transpile for IonQ backend
    t_qc = transpile(qc, backend=backend)
    
    # Submit job
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    
    # Get counts, returns a dictionary with the key equal to the result and the value equal to the number of times it ocurred
    counts = result.get_counts()
    
    # For single shot, return the measured outcome
    if shots == 1:
        # Convert from binary string to list of bits
        outcome = list(counts.keys())[0]  # Get the single outcome
        return [int(bit) for bit in outcome[::-1]]  # Reverse for Qiskit ordering
    
    # For multiple shots, return most frequent outcome
    most_frequent = max(counts, key=counts.get) # finds which key has more occurences
    return [int(bit) for bit in most_frequent[::-1]]

def eve_intercept_resend_batch(alice_bits: List[int], alice_bases: List[int], eve_bases: List[int]) -> QuantumCircuit:
    """Eve intercepts and resends using batch processing"""
    # Eve measures in her bases
    alice_circuit = prepare_batch_circuit(alice_bits, alice_bases)
    eve_measure_circuit = measure_batch_circuit(alice_circuit, eve_bases)
    eve_measurements = run_circuit_get_bits(eve_measure_circuit)
    
    # Eve re-prepares qubits in her measured states
    return prepare_batch_circuit(eve_measurements, eve_bases)

def sift_key(alice_bits: List[int], alice_bases: List[int], bob_bits: List[int], bob_bases: List[int]) -> Tuple[List[int], List[int], List[int]]:
    """Return (sifted_alice, sifted_bob, indices_kept) where we keep positions with matching bases."""
    sifted_a = []
    sifted_b = []
    indices = []
    for i, (abits, abases, bbits, bbases) in enumerate(zip(alice_bits, alice_bases, bob_bits, bob_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_a.append(alice_bits[i])
            sifted_b.append(bob_bits[i])
            indices.append(i)
    return sifted_a, sifted_b, indices

def error_rate(a_bits: List[int], b_bits: List[int]) -> float:
    if not a_bits:
        return 0.0
    mismatches = sum(x != y for x, y in zip(a_bits, b_bits))
    return mismatches / len(a_bits)

def run_bb84_ionq(n: int = 8, with_eve: bool = False, verbose: bool = True):
    """Run BB84 on IonQ - note smaller n due to hardware limitations"""
    
    if n > 11 and flag:  # IonQ currently has limited qubits
        print(f"Warning: Reducing n from {n} to 8 due to hardware limitations")
        n = 8
    
    # 1) Alice chooses bits and bases
    alice_bits = random_bits(n)
    alice_bases = random_bases(n)

    # 2) Prepare circuits
    if with_eve:
        eve_bases = random_bases(n)
        # Eve intercepts and resends
        forwarded_circuit = eve_intercept_resend_batch(alice_bits, alice_bases, eve_bases)
    else:
        # No Eve - use Alice's original preparation
        forwarded_circuit = prepare_batch_circuit(alice_bits, alice_bases)

    # 3) Bob chooses measurement bases and measures
    bob_bases = random_bases(n)
    bob_measure_circuit = measure_batch_circuit(forwarded_circuit, bob_bases)
    
    # 4) Run on IonQ
    bob_bits = run_circuit_get_bits(bob_measure_circuit)

    # 5) Sift keys
    sifted_a, sifted_b, indices = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    
    # 6) Calculate error rate
    err = error_rate(sifted_a, sifted_b)

    if verbose:
        print(f"Using IonQ backend: {backend.name}")
        print(f"Alice bits:      {alice_bits}")
        print(f"Alice bases:     {alice_bases}")
        print(f"Bob bits:        {bob_bits}")
        print(f"Bob bases:       {bob_bases}")
        print(f"Sifted indices:  {indices}")
        print(f"Sifted Alice:    {sifted_a}")
        print(f"Sifted Bob:      {sifted_b}")
        print(f"Error rate:      {err:.3f}")
        print(f"Key length:      {len(sifted_a)}")

    return {
        'alice_bits': alice_bits,
        'alice_bases': alice_bases,
        'bob_bits': bob_bits,
        'bob_bases': bob_bases,
        'sifted_alice': sifted_a,
        'sifted_bob': sifted_b,
        'sifted_indices': indices,
        'error_rate': err
    }

# Test the IonQ version
print("Testing BB84 on IonQ Simulator:")
result = run_bb84_ionq(n=4, with_eve=False)

Testing BB84 on IonQ Simulator:
Using IonQ backend: ionq_simulator
Alice bits:      [0, 0, 1, 0]
Alice bases:     [0, 1, 0, 1]
Bob bits:        [0, 0, 0, 0]
Bob bases:       [0, 1, 1, 1]
Sifted indices:  [0, 1, 3]
Sifted Alice:    [0, 0, 0]
Sifted Bob:      [0, 0, 0]
Error rate:      0.000
Key length:      3
